# Projeto SIM - Grupo 02 (Carregamento e Preparação de Dados - PC2)

**Fonte Oficial dos Dados:** Ministério da Saúde / DATASUS (`ftp://ftp.datasus.gov.br`)  
**Unidade Federativa:** Bahia (BA) | **Período de Análise:** 2022 a 2025

### 1. Origem, Diretórios do Servidor e Status dos Dados
Todos os microdados e tabelas auxiliares são extraídos diretamente do servidor FTP oficial do DATASUS durante a execução deste notebook, garantindo total reprodutibilidade sem necessidade de upload manual de arquivos:

* **`DOBA2022.dbc`** — Diretório FTP: `/dissemin/publicos/SIM/CID10/DORES` *(Dados Definitivos - 2022)*
* **`DOBA2023.dbc`** — Diretório FTP: `/dissemin/publicos/SIM/CID10/DORES` *(Dados Definitivos - 2023)*
* **`DOBA2024.dbc`** — Diretório FTP: `/dissemin/publicos/SIM/CID10/DORES` *(Dados Definitivos - 2024)*
* **`DOBA2025.dbc`** — Diretório FTP: `/dissemin/publicos/SIM/PRELIM/DORES` *(**Dados Preliminares** - 2025, sujeitos a revisão pelo Ministério da Saúde)*
* **Documentação Oficial (Dicionário):** Diretório FTP: `/dissemin/publicos/SIM/CID10/DOCS` (Estrutura do SIM distribuída pelo DATASUS).
* **Base Territorial (Regiões de Saúde - M3):** Diretório FTP: `/territorio/tabelas/2026`, contendo o arquivo (`06-base_territorial_jun26`), este arquivo contém o relacionamento entre municípios (`rl_municip_regsaud.dbf`) e as 28 Regiões de Saúde da Bahia (`tb_regsaud.dbf`).

*(Nota: As datas exatas de última modificação/atualização de cada arquivo no servidor do DATASUS são consultadas via protocolo FTP e exibidas automaticamente na Célula 3).*

In [1]:
!pip install -q pyreaddbc dbfread pandas

import ftplib
import os
import zipfile
import glob
import shutil
from datetime import datetime
import pandas as pd
from pyreaddbc import dbc2dbf
from dbfread import DBF
from google.colab import drive

# 1. Montar o Google Drive para salvar os artefatos para o PC3
drive.mount('/content/drive')

PASTA_DRIVE = '/content/drive/MyDrive/Drive-Grupo02'
os.makedirs(PASTA_DRIVE, exist_ok=True)
print(f"Diretório de exportação no Google Drive configurado: {PASTA_DRIVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Diretório de exportação no Google Drive configurado: /content/drive/MyDrive/Drive-Grupo02


In [2]:
arquivos_alvo = [
    {'nome': 'DOBA2022.dbc', 'ano': '2022', 'pasta': '/dissemin/publicos/SIM/CID10/DORES', 'status': 'Definitivo'},
    {'nome': 'DOBA2023.dbc', 'ano': '2023', 'pasta': '/dissemin/publicos/SIM/CID10/DORES', 'status': 'Definitivo'},
    {'nome': 'DOBA2024.dbc', 'ano': '2024', 'pasta': '/dissemin/publicos/SIM/CID10/DORES', 'status': 'Definitivo'},
    {'nome': 'DOBA2025.dbc', 'ano': '2025', 'pasta': '/dissemin/publicos/SIM/PRELIM/DORES', 'status': 'Preliminar'}
]

SALVAR_CSV_BRUTO_INDIVIDUAL = True

ftp = ftplib.FTP('ftp.datasus.gov.br', timeout=60)
ftp.login()

lista_dfs = []
metadados_arquivos = []

print("1. Baixando microdados do SIM no FTP do DATASUS (Leitura Única)...")
for arq in arquivos_alvo:
    nome_dbc = arq['nome']
    pasta_ftp = arq['pasta']
    nome_dbf = nome_dbc.replace('.dbc', '.dbf')

    ftp.cwd(pasta_ftp)

    # Captura a data de atualização oficial do arquivo no servidor FTP
    try:
        resp_mdtm = ftp.sendcmd(f'MDTM {nome_dbc}')
        codigo_data = resp_mdtm.split()[1]
        data_atualizacao = datetime.strptime(codigo_data[:8], '%Y%m%d').strftime('%d/%m/%Y')
    except Exception:
        data_atualizacao = "Verificada no FTP"

    print(f" -> Baixando {nome_dbc} ({arq['status']} | Atualizado em: {data_atualizacao})...")
    with open(nome_dbc, 'wb') as f:
        ftp.retrbinary(f'RETR {nome_dbc}', f.write)

    dbc2dbf(nome_dbc, nome_dbf)
    tabela = DBF(nome_dbf, encoding='iso-8859-1', load=True)

    df_temp = pd.DataFrame(iter(tabela)).astype(str)
    df_temp.columns = df_temp.columns.str.strip().str.upper()

    if SALVAR_CSV_BRUTO_INDIVIDUAL:
        nome_csv_ind = nome_dbc.replace('.dbc', '_bruto.csv')
        df_temp.to_csv(nome_csv_ind, index=False, sep=',', encoding='utf-8')

    # Registra o ano do arquivo e o ano extraído da data do óbito
    df_temp['ANO_ARQUIVO'] = arq['ano']
    df_temp['ANO_OBITO'] = df_temp['DTOBITO'].str[-4:]

    metadados_arquivos.append({
        'Arquivo': nome_dbc,
        'Diretório FTP': pasta_ftp,
        'Status': arq['status'],
        'Última Atualização': data_atualizacao,
        'Linhas Brutas': df_temp.shape[0],
        'Colunas Originais': df_temp.shape[1] - 2
    })

    lista_dfs.append(df_temp)
    os.remove(nome_dbc)
    os.remove(nome_dbf)

df_bruto = pd.concat(lista_dfs, ignore_index=True)

print("\nResumo de Origem e Atualização dos Arquivos:")
display(pd.DataFrame(metadados_arquivos))

# Validação rigorosa da base bruta (423.233 registros)
total_bruto = df_bruto.shape[0]
print(f"\nTotal de registros brutos consolidados: {total_bruto}")
assert total_bruto == 423233, f"ERRO DE VALIDAÇÃO: Esperado 423.233 registros brutos, mas foram encontrados {total_bruto}."

# Cálculo do preenchimento sobre a BASE BRUTA (para bater com a ficha técnica)
print("Calculando resumo de preenchimento sobre os arquivos brutos...")
colunas_originais = [c for c in df_bruto.columns if c not in ['ANO_ARQUIVO', 'ANO_OBITO']]
preenchimento_bruto = (df_bruto[colunas_originais].replace(['', 'None', 'nan', 'NaN'], pd.NA).notna().mean() * 100).round(2)
preenchimento_bruto.to_csv('resumo_preenchimento.csv', header=['PREENCHIMENTO_PERCENTUAL'])
shutil.copy('resumo_preenchimento.csv', os.path.join(PASTA_DRIVE, 'resumo_preenchimento.csv'))

# 2. Download do Dicionário Oficial na pasta /dissemin/publicos/SIM/CID10/DOCS
print("\n2. Baixando Dicionário/Estrutura oficial no FTP (/dissemin/publicos/SIM/CID10/DOCS)...")
ftp.cwd('/dissemin/publicos/SIM/CID10/DOCS')
arquivos_docs = ftp.nlst()

nome_doc_alvo = 'Estrutura_do_SIM_2025.pdf'

# Localiza o arquivo exato ignorando diferenças de maiúsculas/minúsculas no servidor FTP
encontrados = [f for f in arquivos_docs if f.lower() == nome_doc_alvo.lower()]

if encontrados:
    nome_doc_ftp = encontrados[0]
else:
    # Fallback seguro: busca apenas arquivos que contenham 'ESTRUTURA' no nome
    candidatos_estrutura = [f for f in arquivos_docs if 'ESTRUTURA' in f.upper()]
    nome_doc_ftp = sorted(candidatos_estrutura)[-1] if candidatos_estrutura else nome_doc_alvo

with open(nome_doc_ftp, 'wb') as f:
    ftp.retrbinary(f'RETR {nome_doc_ftp}', f.write)

shutil.copy(nome_doc_ftp, os.path.join(PASTA_DRIVE, nome_doc_ftp))
print(f" -> Documento oficial baixado com seu nome original: {nome_doc_ftp}")

# 3. Download e extração direta da Base Territorial oficial no FTP
pasta_territorial = '/territorio/tabelas/2026'
arquivo_zip_territorial = '06-base_territorial_jun26.zip'
tabelas_necessarias = ['rl_municip_regsaud.dbf', 'tb_regsaud.dbf']

print(f"\n3. Baixando Base Territorial de {pasta_territorial}/{arquivo_zip_territorial}...")
ftp.cwd(pasta_territorial)

# Caso o nome no servidor tenha alguma variação de maiúsculas/minúsculas, localiza o arquivo exato
arquivos_no_ftp = ftp.nlst()
if arquivo_zip_territorial not in arquivos_no_ftp:
    candidatos_zip = [f for f in arquivos_no_ftp if 'base_territorial' in f.lower() and f.lower().endswith('.zip')]
    if candidatos_zip:
        arquivo_zip_territorial = candidatos_zip[0]

with open(arquivo_zip_territorial, 'wb') as f:
    ftp.retrbinary(f'RETR {arquivo_zip_territorial}', f.write)

ftp.quit()

# Extrai apenas os dois arquivos .dbf necessários (independente de estarem em subpastas dentro do .zip)
print(f" -> Extraindo {tabelas_necessarias} de {arquivo_zip_territorial}...")
with zipfile.ZipFile(arquivo_zip_territorial, 'r') as z:
    for membro in z.namelist():
        nome_base = os.path.basename(membro).lower()
        if nome_base in tabelas_necessarias:
            with open(nome_base, 'wb') as out_f:
                out_f.write(z.read(membro))
            # Salva uma cópia de segurança na pasta do Google Drive
            shutil.copy(nome_base, os.path.join(PASTA_DRIVE, nome_base))
            print(f" -> '{nome_base}' extraído com sucesso!")

# Remove o .zip temporário para manter o Colab limpo
os.remove(arquivo_zip_territorial)

assert os.path.exists('rl_municip_regsaud.dbf') and os.path.exists('tb_regsaud.dbf'), (
    "ERRO CRÍTICO: As tabelas 'rl_municip_regsaud.dbf' e 'tb_regsaud.dbf' não foram encontradas dentro do .zip."
)
print("✅ Etapa de download e validação bruta concluída com sucesso!")

1. Baixando microdados do SIM no FTP do DATASUS (Leitura Única)...
 -> Baixando DOBA2022.dbc (Definitivo | Atualizado em: 21/12/2023)...
 -> Baixando DOBA2023.dbc (Definitivo | Atualizado em: 19/12/2024)...
 -> Baixando DOBA2024.dbc (Definitivo | Atualizado em: 23/12/2025)...
 -> Baixando DOBA2025.dbc (Preliminar | Atualizado em: 28/08/2026)...

Resumo de Origem e Atualização dos Arquivos:


,Arquivo,Diretório FTP,Status,Última Atualização,Linhas Brutas,Colunas Originais
0,DOBA2022.dbc,/dissemin/publicos/SIM/CID10/DORES,Definitivo,21/12/2023,107641,87
1,DOBA2023.dbc,/dissemin/publicos/SIM/CID10/DORES,Definitivo,19/12/2024,103081,87
2,DOBA2024.dbc,/dissemin/publicos/SIM/CID10/DORES,Definitivo,23/12/2025,106212,87
3,DOBA2025.dbc,/dissemin/publicos/SIM/PRELIM/DORES,Preliminar,28/08/2026,106299,87



Total de registros brutos consolidados: 423233
Calculando resumo de preenchimento sobre os arquivos brutos...

2. Baixando Dicionário/Estrutura oficial no FTP (/dissemin/publicos/SIM/CID10/DOCS)...
 -> Documento oficial baixado com seu nome original: Estrutura_do_SIM_2025.pdf

3. Baixando Base Territorial de /territorio/tabelas/2026/06-base_territorial_jun26.zip...
 -> Extraindo ['rl_municip_regsaud.dbf', 'tb_regsaud.dbf'] de 06-base_territorial_jun26.zip...
 -> 'tb_regsaud.dbf' extraído com sucesso!
 -> 'rl_municip_regsaud.dbf' extraído com sucesso!
✅ Etapa de download e validação bruta concluída com sucesso!


In [ ]:
print("1. Aplicando recortes definidos na ficha técnica...")

assert (df_bruto['ANO_OBITO'] == df_bruto['ANO_ARQUIVO']).all(), "ERRO: ANO_OBITO diverge do ANO_ARQUIVO!"

# Recorte 1: Manter apenas residentes da Bahia (códigos iniciados em 29)
df_estudo = df_bruto[df_bruto['CODMUNRES'].str.startswith('29', na=False)].copy()

# Recorte 2: Excluir óbitos com idade ignorada e menores de 1 ano
df_estudo = df_estudo[df_estudo['IDADE'].str.match('^[45]', na=False)].copy()

# Recorte 3: Criar a Variável-Alvo (Causas Externas = capítulos V, W, X, Y da CID-10)
df_estudo['ALVO_CAUSA_EXTERNA'] = df_estudo['CAUSABAS'].str[0].isin(['V', 'W', 'X', 'Y']).astype(int)

# Validação rigorosa de linhas após filtragem (412.692 linhas)
linhas_filtradas = df_estudo.shape[0]
print(f" -> Registros após filtros da ficha técnica: {linhas_filtradas}")
assert linhas_filtradas == 412692, f"ERRO: Esperado exatamente 412.692 linhas, mas obteve {linhas_filtradas}."

# Validação rigorosa do total de Causas Externas (53.800 óbitos)
total_alvo = df_estudo['ALVO_CAUSA_EXTERNA'].sum()
prop_alvo = (total_alvo / linhas_filtradas * 100).round(2)
print(f" -> Total de Causas Externas (ALVO = 1): {total_alvo} ({prop_alvo}%)")
assert total_alvo == 53800, f"ERRO: Esperado 53.800 causas externas, mas obteve {total_alvo}."

print("\n2. Integrando Regiões de Saúde (Conjunto M3)...")

tab_municipio = DBF('rl_municip_regsaud.dbf', encoding='iso-8859-1', load=True)
df_municipios = pd.DataFrame(iter(tab_municipio)).astype(str)
df_municipios.columns = df_municipios.columns.str.strip().str.upper()

tab_regiao = DBF('tb_regsaud.dbf', encoding='iso-8859-1', load=True)
df_regioes = pd.DataFrame(iter(tab_regiao)).astype(str)
df_regioes.columns = df_regioes.columns.str.strip().str.upper()

# Identifica dinamicamente a coluna de nome da região na tb_regsaud (evita erro caso não se chame DS_NOME)
colunas_nome_candidatas = [c for c in df_regioes.columns if c != 'CO_REGSAUD' and any(k in c for k in ['NOME', 'DS_', 'NO_', 'DESC', 'REG'])]
col_nome_regiao = colunas_nome_candidatas[0] if colunas_nome_candidatas else [c for c in df_regioes.columns if c != 'CO_REGSAUD'][0]
print(f" -> Coluna de nome da região identificada em tb_regsaud: '{col_nome_regiao}'")

# Padroniza chave de 6 dígitos do município e REMOVE DUPLICATAS antes do merge para não duplicar óbitos
df_municipios['MUNICIPIO_6'] = df_municipios['CO_MUNICIP'].str.strip().str[:6]
df_municipios = df_municipios[['MUNICIPIO_6', 'CO_REGSAUD']].drop_duplicates(subset=['MUNICIPIO_6'])

df_regioes['CO_REGSAUD'] = df_regioes['CO_REGSAUD'].str.strip()
df_regioes = df_regioes[['CO_REGSAUD', col_nome_regiao]].drop_duplicates(subset=['CO_REGSAUD'])

df_estudo['CODMUNRES_6'] = df_estudo['CODMUNRES'].str.strip().str[:6]

# 1º Merge: Município -> Código da Região de Saúde
df_estudo = df_estudo.merge(
    df_municipios,
    left_on='CODMUNRES_6',
    right_on='MUNICIPIO_6',
    how='left'
)

# 2º Merge: Código da Região -> Nome da Região de Saúde (NO_REGSAUD)
df_estudo = df_estudo.merge(
    df_regioes,
    on='CO_REGSAUD',
    how='left'
).rename(columns={col_nome_regiao: 'NO_REGSAUD'})

df_estudo = df_estudo.drop(columns=['MUNICIPIO_6', 'CODMUNRES_6', 'CO_REGSAUD'])

# Validações pós-merge exigidas pela revisão:
assert df_estudo.shape[0] == 412692, f"ERRO PÓS-MERGE: O número de linhas mudou para {df_estudo.shape[0]}!"
assert 'NO_REGSAUD' in df_estudo.columns, "ERRO: A coluna NO_REGSAUD não foi criada!"

qtd_regioes = df_estudo['NO_REGSAUD'].dropna().nunique()
sem_regiao = df_estudo['NO_REGSAUD'].isna().sum()
cod_290000 = (df_estudo['CODMUNRES'].str[:6] == '290000').sum()

print(f" -> Total de Regiões de Saúde distintas mapeadas na Bahia: {qtd_regioes}")
print(f" -> Óbitos sem região associada: {sem_regiao} (Óbitos com município ignorado '290000': {cod_290000} — tratamento ficará para o PC3)")

assert qtd_regioes == 28, f"ERRO: Esperadas as 28 Regiões de Saúde da Bahia, mas foram encontradas {qtd_regioes}."
print("✅ Cruzamento territorial concluído e validado com sucesso!")

1. Aplicando recortes definidos na ficha técnica...
 -> Registros após filtros da ficha técnica: 412692
 -> Total de Causas Externas (ALVO = 1): 53800 (13.04%)

2. Integrando Regiões de Saúde (Conjunto M3)...
 -> Coluna de nome da região identificada em tb_regsaud: 'DS_NOME'
 -> Total de Regiões de Saúde distintas mapeadas na Bahia: 28
 -> Óbitos sem região associada: 532 (Óbitos com município ignorado '290000': 532 — tratamento ficará para o PC3)
✅ Cruzamento territorial concluído e validado com sucesso!


In [ ]:
colunas_estudo = [
    'ANO_OBITO',          # Ano extraído de DTOBITO (Controle temporal)
    'CODMUNRES',          # Código IBGE do Município de Residência (6 dígitos)
    'NO_REGSAUD',         # M3: Nome da Região de Saúde (28 regiões da Bahia)
    'IDADE',              # M1: Idade codificada padrão SIM
    'SEXO',               # M1: Sexo
    'RACACOR',            # M2: Raça/Cor
    'ESTCIV',             # M2: Situação Conjugal
    'ESC2010',            # M2: Escolaridade (padrão 2010)
    'CAUSABAS',           # CID-10 da Causa Básica (conferência)
    'ALVO_CAUSA_EXTERNA'  # Variável-Alvo binária (1 = Capítulo XX - Causas Externas; 0 = Demais)
]

# Validação extra do ANO_OBITO conforme PC3 (feita antes de remover ANO_ARQUIVO)
if 'ANO_ARQUIVO' in df_estudo.columns:
    assert (df_estudo['ANO_OBITO'] == df_estudo['ANO_ARQUIVO']).all(), "ERRO: ANO_OBITO diverge do ANO_ARQUIVO em alguns registros!"

df_estudo = df_estudo[colunas_estudo]

print("=" * 75)
print("RELATÓRIO DE DIMENSÕES E ESTRUTURA DAS BASES (ETAPA 4 - PC2)")
print("=" * 75)

# Exibe o resumo de preenchimento bruto calculado anteriormente
print("\n• Resumo de Preenchimento da Base Bruta (Ficha Técnica):")
# Filtramos apenas as colunas de interesse do quadro 3 da ficha técnica para visualização
vars_ficha = ['RACACOR', 'ESC2010', 'ESTCIV', 'CIRCOBITO', 'ATESTANTE']
if 'preenchimento_bruto' in locals():
    display(preenchimento_bruto.loc[preenchimento_bruto.index.isin(vars_ficha)])

# 1. Dimensões da base bruta vs. base do estudo
qtd_cols_originais = df_bruto.shape[1] - 2  # Desconta ANO_ARQUIVO e ANO_OBITO adicionados na leitura
print(f"\n• Dimensões da Base Bruta (2022-2025):     {df_bruto.shape[0]:,} linhas x {qtd_cols_originais} colunas originais ({df_bruto.shape[1]} com colunas auxiliares)")
print(f"• Dimensões da Base de Estudo (Filtrada):  {df_estudo.shape[0]:,} linhas x {df_estudo.shape[1]} colunas")

# 2. Lista de atributos com seus tipos e descrições (Item 3 da Etapa 4)
descricoes_atributos = {
    'ANO_OBITO': 'Controle Temporal — Ano extraído da data do óbito (DTOBITO)',
    'CODMUNRES': 'Territorial — Código IBGE (6 dígitos) do município de residência na Bahia',
    'NO_REGSAUD': 'Conjunto M3 — Nome da Região de Saúde da Bahia (28 regiões)',
    'IDADE': 'Conjunto M1 — Idade codificada do falecido (primeiro dígito 4=anos, 5=mais de 100 anos)',
    'SEXO': 'Conjunto M1 — Sexo do falecido',
    'RACACOR': 'Conjunto M2 — Raça/Cor',
    'ESTCIV': 'Conjunto M2 — Situação Conjugal',
    'ESC2010': 'Conjunto M2 — Escolaridade (padrão 2010)',
    'CAUSABAS': 'Controle — Código CID-10 da causa básica do óbito (A ser removido para ML)',
    'ALVO_CAUSA_EXTERNA': 'Variável-Alvo — Causa Externa (1 = Capítulo XX [V01-Y98]; 0 = Outras causas)'
}

# Tratamento de nulos/vazios para a contagem correta na tabela de atributos
df_estudo_na = df_estudo.replace(['', ' ', 'None', 'nan', 'NaN'], pd.NA)
df_estudo_na['RACACOR'] = df_estudo_na['RACACOR'].replace('9', pd.NA)
df_estudo_na['ESC2010'] = df_estudo_na['ESC2010'].replace('9', pd.NA)
df_estudo_na['ESTCIV'] = df_estudo_na['ESTCIV'].replace('9', pd.NA)
df_estudo_na['SEXO'] = df_estudo_na['SEXO'].replace(['0', '9', 'I'], pd.NA)
df_estudo_na['CODMUNRES'] = df_estudo_na['CODMUNRES'].replace('290000', pd.NA)

tabela_atributos = pd.DataFrame({
    'Atributo': df_estudo.columns,
    'Tipo no Pandas (dtype)': [str(df_estudo[c].dtype) for c in df_estudo.columns],
    'Não-Nulos': [df_estudo_na[c].notna().sum() for c in df_estudo.columns],
    'Papel / Descrição no Estudo': [descricoes_atributos[c] for c in df_estudo.columns]
})
print("\n• Lista de Atributos da Base de Estudo e seus Tipos (Item 3 da Etapa 4):")
display(tabela_atributos)

# 3. Contagem de óbitos por ano (Bruta vs. Estudo)
contagem_ano = pd.DataFrame({
    'Registros na Base Bruta': df_bruto['ANO_ARQUIVO'].value_counts().sort_index(),
    'Registros na Base de Estudo': df_estudo['ANO_OBITO'].value_counts().sort_index(),
    'Causas Externas (Alvo = 1)': df_estudo.groupby('ANO_OBITO')['ALVO_CAUSA_EXTERNA'].sum(),
    '% Causas Externas': (df_estudo.groupby('ANO_OBITO')['ALVO_CAUSA_EXTERNA'].mean() * 100).round(2)
})
print("\n• Contagem de Registros por Ano:")
display(contagem_ano)

# CUIDADO PC3: Remover CAUSABAS antes de salvar para evitar vazamento de dados
df_estudo_export = df_estudo.drop(columns=['CAUSABAS'])
print(f"\n• Dimensões da Base Exportada (Sem CAUSABAS): {df_estudo_export.shape[0]:,} linhas x {df_estudo_export.shape[1]} colunas")

# 4. Exportação padronizada (separador ',' e nome sim_BA_2022_2025_base_estudo.csv.gz)
nome_arquivo_final = 'sim_BA_2022_2025_base_estudo.csv.gz'
caminho_drive_final = os.path.join(PASTA_DRIVE, nome_arquivo_final)

print(f"\nExportando '{nome_arquivo_final}' (separador=',')...")
df_estudo_export.to_csv(nome_arquivo_final, index=False, sep=',', encoding='utf-8', compression='gzip')
shutil.copy(nome_arquivo_final, caminho_drive_final)

print(f"✅ Arquivo salvo localmente no Colab: /content/{nome_arquivo_final}")
print(f"✅ Cópia persistente salva no Google Drive: {caminho_drive_final}")

RELATÓRIO DE DIMENSÕES E ESTRUTURA DAS BASES (ETAPA 4 - PC2)

• Resumo de Preenchimento da Base Bruta (Ficha Técnica):


,0
RACACOR,96.84
ESTCIV,94.48
ESC2010,92.82
CIRCOBITO,12.81
ATESTANTE,89.15



• Dimensões da Base Bruta (2022-2025):     423,233 linhas x 87 colunas originais (89 com colunas auxiliares)
• Dimensões da Base de Estudo (Filtrada):  412,692 linhas x 10 colunas

• Lista de Atributos da Base de Estudo e seus Tipos (Item 3 da Etapa 4):


,Atributo,Tipo no Pandas (dtype),Não-Nulos,Papel / Descrição no Estudo
0,ANO_OBITO,object,412692,Controle Temporal — Ano extraído da data do ób...
1,CODMUNRES,object,412160,Territorial — Código IBGE (6 dígitos) do munic...
2,NO_REGSAUD,object,412160,Conjunto M3 — Nome da Região de Saúde da Bahia...
3,IDADE,object,412692,Conjunto M1 — Idade codificada do falecido (pr...
4,SEXO,object,412667,Conjunto M1 — Sexo do falecido
5,RACACOR,object,400671,Conjunto M2 — Raça/Cor
6,ESTCIV,object,370950,Conjunto M2 — Situação Conjugal
7,ESC2010,object,338454,Conjunto M2 — Escolaridade (padrão 2010)
8,CAUSABAS,object,412692,Controle — Código CID-10 da causa básica do ób...
9,ALVO_CAUSA_EXTERNA,int64,412692,Variável-Alvo — Causa Externa (1 = Capítulo XX...



• Contagem de Registros por Ano:


,Registros na Base Bruta,Registros na Base de Estudo,Causas Externas (Alvo = 1),% Causas Externas
2022,107641,104798,13506,12.89
2023,103081,100380,13623,13.57
2024,106212,103690,13728,13.24
2025,106299,103824,12943,12.47



• Dimensões da Base Exportada (Sem CAUSABAS): 412,692 linhas x 9 colunas

Exportando 'sim_BA_2022_2025_base_estudo.csv.gz' (separador=',')...
✅ Arquivo salvo localmente no Colab: /content/sim_BA_2022_2025_base_estudo.csv.gz
✅ Cópia persistente salva no Google Drive: /content/drive/MyDrive/Drive-Grupo02/sim_BA_2022_2025_base_estudo.csv.gz


In [ ]:
import shutil
import glob
import os

print("Copiando arquivos brutos individuais para o Google Drive...")
arquivos_brutos = glob.glob('*_bruto.csv')

if arquivos_brutos:
    for arq in arquivos_brutos:
        caminho_destino = os.path.join(PASTA_DRIVE, arq)
        shutil.copy(arq, caminho_destino)
        print(f" -> Arquivo {arq} salvo em: {caminho_destino}")
    print("\n✅ Todos os arquivos brutos individuais foram salvos no Drive com sucesso!")
else:
    print("Nenhum arquivo bruto encontrado no diretório atual.")

Copiando arquivos brutos individuais para o Google Drive...
 -> Arquivo DOBA2025_bruto.csv salvo em: /content/drive/MyDrive/Drive-Grupo02/DOBA2025_bruto.csv
 -> Arquivo DOBA2022_bruto.csv salvo em: /content/drive/MyDrive/Drive-Grupo02/DOBA2022_bruto.csv
 -> Arquivo DOBA2024_bruto.csv salvo em: /content/drive/MyDrive/Drive-Grupo02/DOBA2024_bruto.csv
 -> Arquivo DOBA2023_bruto.csv salvo em: /content/drive/MyDrive/Drive-Grupo02/DOBA2023_bruto.csv

✅ Todos os arquivos brutos individuais foram salvos no Drive com sucesso!
